# MecâniQA — OAT 3 | Encontro 30/09: Tunando o Meu Modelo

**Time:** Boa Vista — Alex Oliveira Santos, Alice Gomes Aragão, Ana Clara Ribeiro da Silva, Lorran Edley Matos Ribeiro, Rodrigo dos Anjos Lamartine

**Objetivo prático:** implementar e avaliar algoritmos de regressão com hiperparâmetros padrão e verificar se os modelos "crus" já batem os baselines da oficina.

## Decisões da equipe (brainstorm)

**1. Escolha do algoritmo:** Random Forest e Gradient Boosting, com Árvore de Decisão e Regressão Linear como referência. Modelos de árvores capturam padrões não lineares, como o pico de demanda no fim de semana e a tendência de crescimento da oficina, sem precisar transformar os dados. O Random Forest treina várias árvores e tira a média das previsões, o que reduz a variância. O Gradient Boosting treina árvores em sequência, e cada uma corrige o erro da anterior.

**2. Linha do tempo:** a divisão é cronológica, sem embaralhar os dias: os primeiros 80% para treino e os 20% finais para teste. O `fit()` usa somente o treino, e o teste só entra no `predict()` para medir o erro. As features de lag e rolling usam `shift`, então cada linha só carrega informação do passado.

## 0. Preparação: importações, dados limpos e features (OAT 1 e 2)

In [1]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

SEED = 42
ALVOS = ["Trocas_Oleo", "Manutencao_Motor"]
NOMES = {"Trocas_Oleo": "Trocas de Óleo", "Manutencao_Motor": "Manutenção de Motor"}
pd.options.display.float_format = "{:.2f}".format

# --- Dataset limpo (pipeline da OAT 1) ---
df = pd.read_excel("mecaniqa_dataset.xlsx")
df["Data"] = pd.to_datetime(df["Data"])
df = df.set_index("Data").sort_index().asfreq("D")

for alvo in ALVOS:
    df.loc[df[alvo] < 0, alvo] = np.nan                  # negativos são inválidos
    df[alvo] = df[alvo].interpolate(method="time")       # ausentes e inválidos
    q1, q3 = df[alvo].quantile([0.25, 0.75])
    iqr = q3 - q1
    df[alvo] = df[alvo].clip(lower=max(0, q1 - 1.5 * iqr), upper=q3 + 1.5 * iqr)

# --- Features temporais (OAT 2): lags 1, 7, 30 e rolling mean de 7 dias (só passado) ---
df_feat = df.copy()
for alvo in ALVOS:
    for lag in [1, 7, 30]:
        df_feat[f"{alvo}_lag_{lag}"] = df_feat[alvo].shift(lag)
    df_feat[f"{alvo}_rolling_mean_7"] = df_feat[alvo].rolling(7, min_periods=7).mean().shift(1)

df_feat = df_feat.dropna()
FEATURES = [c for c in df_feat.columns if c not in ALVOS]
print(f"Dataset de features: {df_feat.shape[0]} dias, {len(FEATURES)} features")

Dataset de features: 701 dias, 8 features


## 1. Divisão temporal (treino no passado, teste no futuro)

In [2]:
n_treino = int(len(df_feat) * 0.80)
treino, teste = df_feat.iloc[:n_treino], df_feat.iloc[n_treino:]
X_treino, X_teste = treino[FEATURES], teste[FEATURES]

assert treino.index.max() < teste.index.min(), "O treino precisa terminar antes do teste!"
print(f"Treino: {treino.index.min().date()} a {treino.index.max().date()}  ({len(treino)} dias)")
print(f"Teste : {teste.index.min().date()} a {teste.index.max().date()}  ({len(teste)} dias)")

Treino: 2024-01-31 a 2025-08-12  (560 dias)
Teste : 2025-08-13 a 2025-12-31  (141 dias)


## 2. Baselines antigos (referência a ser batida) e função de erro

In [3]:
def avaliar_metricas(y_real, y_previsto):
    y_real, y_prev = np.asarray(y_real, dtype=float), np.asarray(y_previsto, dtype=float)
    mascara = y_real != 0   # MAPE não é definido quando o valor real é zero
    return {
        "RMSE": np.sqrt(mean_squared_error(y_real, y_prev)),
        "MAPE": mean_absolute_percentage_error(y_real[mascara], y_prev[mascara]) * 100,
        "MAE": mean_absolute_error(y_real, y_prev),
    }


# Baselines da OAT 2 (shift(1) evita olhar o dia atual)
def previsao_naive(serie):
    return serie.shift(1)

def previsao_media_movel(serie, janela=7):
    return serie.rolling(janela, min_periods=janela).mean().shift(1)


# Baseline antigo de cada variável = o de menor MAE no TREINO (sem olhar o teste)
baseline_prev, baseline_nome = {}, {}
for alvo in ALVOS:
    candidatos = {"Naive": previsao_naive(df[alvo]), "Média Móvel 7d": previsao_media_movel(df[alvo])}
    mae_treino = {n: avaliar_metricas(df[alvo].loc[treino.index], p.loc[treino.index])["MAE"] for n, p in candidatos.items()}
    baseline_nome[alvo] = min(mae_treino, key=mae_treino.get)
    baseline_prev[alvo] = candidatos[baseline_nome[alvo]].loc[teste.index]
    print(f"{NOMES[alvo]}: baseline antigo = {baseline_nome[alvo]}")

Trocas de Óleo: baseline antigo = Naive
Manutenção de Motor: baseline antigo = Média Móvel 7d


## 3. Passo 1 — Instanciar modelos clássicos de regressão (hiperparâmetros default)

In [4]:
def criar_modelos():
    return {
        "Regressão Linear": LinearRegression(),
        "Árvore de Decisão": DecisionTreeRegressor(random_state=SEED),
        "Random Forest": RandomForestRegressor(random_state=SEED),
        "Gradient Boosting": GradientBoostingRegressor(random_state=SEED),
    }

modelos = {alvo: criar_modelos() for alvo in ALVOS}
print("Modelos instanciados:", list(modelos["Trocas_Oleo"].keys()))

Modelos instanciados: ['Regressão Linear', 'Árvore de Decisão', 'Random Forest', 'Gradient Boosting']


## 4. Passo 2 e 3 — Pipeline: `fit()` só no treino e `predict()` no teste

In [5]:
previsoes = {alvo: {} for alvo in ALVOS}

for alvo in ALVOS:
    for nome, modelo in modelos[alvo].items():
        modelo.fit(X_treino, treino[alvo])                       # fit EXCLUSIVAMENTE no treino
        previsoes[alvo][nome] = modelo.predict(X_teste)          # predict no teste, com hiperparâmetros default

print("Treino e previsões concluídos para todos os modelos.")

Treino e previsões concluídos para todos os modelos.


## 5. Critério de aceite — erro inicial (RMSE e MAPE) dos modelos "crus" contra o baseline antigo

In [6]:
for alvo in ALVOS:
    b = avaliar_metricas(teste[alvo], baseline_prev[alvo])
    print(f"\n=== {NOMES[alvo]} ===")
    print(f"Baseline antigo ({baseline_nome[alvo]}) - RMSE: {b['RMSE']:.2f}, MAPE: {b['MAPE']:.2f}%")
    for nome in modelos[alvo]:
        m = avaliar_metricas(teste[alvo], previsoes[alvo][nome])
        bate = "BATE o baseline" if (m["RMSE"] < b["RMSE"] and m["MAPE"] < b["MAPE"]) else "não bate o baseline"
        print(f"{nome} - RMSE: {m['RMSE']:.2f}, MAPE: {m['MAPE']:.2f}%  ->  {bate}")


=== Trocas de Óleo ===
Baseline antigo (Naive) - RMSE: 8.92, MAPE: 26.80%
Regressão Linear - RMSE: 4.16, MAPE: 14.02%  ->  BATE o baseline
Árvore de Decisão - RMSE: 5.85, MAPE: 20.38%  ->  BATE o baseline
Random Forest - RMSE: 3.81, MAPE: 13.34%  ->  BATE o baseline
Gradient Boosting - RMSE: 4.00, MAPE: 13.70%  ->  BATE o baseline

=== Manutenção de Motor ===
Baseline antigo (Média Móvel 7d) - RMSE: 3.82, MAPE: 41.05%
Regressão Linear - RMSE: 2.42, MAPE: 26.96%  ->  BATE o baseline
Árvore de Decisão - RMSE: 3.90, MAPE: 38.37%  ->  não bate o baseline
Random Forest - RMSE: 2.48, MAPE: 28.40%  ->  BATE o baseline
Gradient Boosting - RMSE: 2.49, MAPE: 28.45%  ->  BATE o baseline


### Tabela-resumo

In [7]:
linhas = []
for alvo in ALVOS:
    b = avaliar_metricas(teste[alvo], baseline_prev[alvo])
    linhas.append({"Variável": NOMES[alvo], "Modelo": f"Baseline antigo ({baseline_nome[alvo]})", "RMSE": b["RMSE"], "MAPE (%)": b["MAPE"], "Bate o baseline?": "referência"})
    for nome in modelos[alvo]:
        m = avaliar_metricas(teste[alvo], previsoes[alvo][nome])
        bate = "SIM" if (m["RMSE"] < b["RMSE"] and m["MAPE"] < b["MAPE"]) else "não"
        linhas.append({"Variável": NOMES[alvo], "Modelo": nome, "RMSE": m["RMSE"], "MAPE (%)": m["MAPE"], "Bate o baseline?": bate})

pd.DataFrame(linhas).set_index(["Variável", "Modelo"])

RMSE  ...  Bate o baseline?
Variável            Modelo                                  ...                  
Trocas de Óleo      Baseline antigo (Naive)           8.92  ...        referência
                    Regressão Linear                  4.16  ...               SIM
                    Árvore de Decisão                 5.85  ...               SIM
                    Random Forest                     3.81  ...               SIM
                    Gradient Boosting                 4.00  ...               SIM
Manutenção de Motor Baseline antigo (Média Móvel 7d)  3.82  ...        referência
                    Regressão Linear                  2.42  ...               SIM
                    Árvore de Decisão                 3.90  ...               não
                    Random Forest                     2.48  ...               SIM
                    Gradient Boosting                 2.49  ...               SIM

[10 rows x 3 columns]

## Conclusão do encontro

- Os modelos clássicos, mesmo **sem nenhum ajuste de hiperparâmetros**, já superam os baselines antigos da oficina em RMSE e MAPE, porque aprendem o ritmo semanal e a tendência que o baseline (que só repete ou suaviza o passado) não captura.
- O **Random Forest** foi o melhor modelo em Trocas de Óleo; em Manutenção de Motor ele fica praticamente empatado com a Regressão Linear.
- A **Árvore de Decisão** isolada é a mais fraca (uma única árvore tende a decorar o treino), o que justifica conjuntos de árvores.
- **Próximo encontro (07/10):** sintonizar os hiperparâmetros com Grid Search integrado ao `TimeSeriesSplit` para tentar reduzir ainda mais o erro.